# RAG System Demo
## Financial Document Question Answering with GPU Acceleration

### 1. Setup and Installation

In [1]:
import sys
sys.path.append('..')

import torch
from pathlib import Path
from loguru import logger

# Check GPU availability
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

GPU Available: False


### 2. Initialize RAG Pipeline

In [2]:
from src.config import get_gpu_optimized_config
from src.main import initialize_pipeline, answer_question

# Initialize with GPU-optimized config
config = get_gpu_optimized_config()
print(f"Using config:")
print(f"  Chunking: {config.chunking.strategy}")
print(f"  LLM: {config.llm.model_name}")
print(f"  Embedding: {config.embedding.model_name}")
print(f"  GPU: {config.use_gpu}")

# Initialize pipeline (this takes a few minutes)
initialize_pipeline(config=config)

2026-02-07 19:52:17.500 | INFO     | src.pipeline.rag_pipeline:__init__:37 - Initializing RAG Pipeline
2026-02-07 19:52:17.501 | INFO     | src.pipeline.rag_pipeline:_initialize_components:44 - Initializing pipeline components...
2026-02-07 19:52:17.502 | INFO     | src.services.embedding_service:__init__:28 - Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
2026-02-07 19:52:17.503 | INFO     | src.services.embedding_service:__init__:29 - Using device: cpu


Using config:
  Chunking: semantic
  LLM: mistralai/Mistral-7B-Instruct-v0.2
  Embedding: sentence-transformers/all-MiniLM-L6-v2
  GPU: False


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-02-07 19:53:43.184 | INFO     | src.services.embedding_service:__init__:39 - Embedding model loaded successfully
2026-02-07 19:53:43.186 | INFO     | src.services.vector_store:__init__:36 - Using FAISS CPU index
2026-02-07 19:53:43.187 | INFO     | src.services.vector_store:_initialize_index:42 - Initializing FAISS index with dimension 384
2026-02-07 19:53:43.188 | INFO     | src.services.retriever:_initialize_reranker:46 - Loading reranker model: cross-encoder/ms-marco-MiniLM-L-6-v2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

2026-02-07 19:55:22.174 | INFO     | src.services.llm_service:_load_model:87 - Loading model...


ImportError: Using `bitsandbytes` 4-bit quantization requires accelerate: `pip install 'accelerate>=1.1.0'`

### 3. Ask Questions

In [ ]:
# Example: Apple revenue question
question = "What was Apple's total revenue for fiscal year 2024?"
result = answer_question(question)

print(f"Question: {question}")
print(f"\nAnswer: {result['answer']}")
print(f"\nSources: {result['sources']}")

In [ ]:
# Example: Tesla question
question = "What types of vehicles does Tesla currently produce and deliver?"
result = answer_question(question)

print(f"Question: {question}")
print(f"\nAnswer: {result['answer']}")
print(f"\nSources: {result['sources']}")

In [ ]:
# Example: Out-of-scope question
question = "What is Tesla's stock price forecast for 2025?"
result = answer_question(question)

print(f"Question: {question}")
print(f"\nAnswer: {result['answer']}")
print(f"\nSources: {result['sources']}")

### 4. Batch Processing

In [ ]:
from src.main import answer_questions_batch

test_questions = [
    {"question_id": 1, "question": "What was Apple's total revenue for fiscal year 2024?"},
    {"question_id": 6, "question": "What was Tesla's total revenue for 2023?"},
    {"question_id": 11, "question": "What is Tesla's stock price forecast for 2025?"}
]

results = answer_questions_batch(test_questions)

for r in results:
    print(f"\nQ{r['question_id']}: {r['answer'][:100]}...")

### 5. Evaluation

In [ ]:
from src.evaluation import RAGEvaluator

# Evaluate results
evaluator = RAGEvaluator()
metrics = evaluator.evaluate(results)

# Print report
evaluator.print_report(metrics)

### 6. Custom Questions

In [ ]:
# Try your own questions!
custom_question = "Your question here"
result = answer_question(custom_question)

print(f"Question: {custom_question}")
print(f"\nAnswer: {result['answer']}")
print(f"\nSources: {result['sources']}")

### 7. Pipeline Statistics

In [ ]:
from src.main import _pipeline

if _pipeline:
    stats = _pipeline.get_pipeline_stats()
    print("Pipeline Statistics:")
    for key, value in stats.items():
        print(f"  {key}: {value}")